### Needed Imports

In [ ]:
#%pip install imbalanced-learn 

In [ ]:
# Modules de manipulation de données
import pandas as pd
import numpy as np

# Modules de visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Dépendance pour le clustering, la normalisation, la modélisation prédictive
import imblearn
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
from imblearn.over_sampling import SMOTE

### Confusion matrix definition

In [ ]:
def plot_conf_matrix(y_true, y_pred, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.xlabel('Prédit')
    plt.ylabel('Réel')
    plt.title(title)
    plt.show()

In [ ]:
def display_report_and_confusion(y_true, y_pred, model_name="Modèle"):
    print(f"📊 Rapport pour {model_name} :\n")
    print(classification_report(y_true, y_pred))
    plot_conf_matrix(y_true, y_pred, title=f"Confusion Matrix - {model_name}")


### Load the data

In [ ]:
columns = ["age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
           "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
           "hours-per-week", "native-country", "income"]

df = pd.read_csv("../data/raw/adult.data", names=columns, na_values=" ?", skipinitialspace=True)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

df.head()


In [ ]:
# Target binary encodage
df["income"] = df["income"].apply(lambda x: 1 if x == ">50K" else 0)

# Encodage des colonnes catégorielles
categorical_cols = df.select_dtypes(include="object").columns
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

df_encoded.head()

### Statistical description: mean, min, max, etc ...

In [ ]:
df.describe()

In [ ]:
#df.describe(include="all")

In [ ]:
plt.figure(figsize=(10, 12))

# Correlation
correlations = df_encoded.corr(numeric_only=True)[["income"]].sort_values(by="income", ascending=False)

# Heatmap plot
sns.heatmap(correlations, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Corrélation des variables avec le revenu (>50K)")
plt.show()


In [ ]:
features = ["age", "education-num", "hours-per-week", "capital-gain", "capital-loss"]
X = df[features]
X_scaled = StandardScaler().fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=0)
df["cluster"] = kmeans.fit_predict(X_scaled)

sns.pairplot(df, vars=features, hue="cluster", palette="tab10")
plt.show()


In [ ]:
# Encoded variables
X = df_encoded.drop("income", axis=1)
y = df_encoded["income"]

# Train/test definition
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Data normalization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


### Selection of the most appropriated model

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=5000),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Support Vector Machine": SVC()#,
    # "Decision Tree": DecisionTreeClassifier(),
    # "KMeans": KMeans(n_clusters=2, random_state=0)
}

def compare_models(X_train, X_test, y_train, y_test, models_dict, sort_metric="F1-weighted"):
    results = []

    for name, model in models_dict.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

        # Rapport complet
        report = classification_report(y_test, y_pred, output_dict=True)

        result = {
            "Model": name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred),
            "Recall": recall_score(y_test, y_pred),
            "F1-class1": report["1"]["f1-score"],
            "F1-macro": report["macro avg"]["f1-score"],
            "F1-weighted": report["weighted avg"]["f1-score"],
            "AUC": roc_auc_score(y_test, y_proba) if y_proba is not None else "N/A"
        }

        results.append(result)

    results_df = pd.DataFrame(results).sort_values(by=sort_metric, ascending=False)

    best_model_name = results_df.iloc[0]["Model"]
    best_model_score = results_df.iloc[0][sort_metric]
    plot_conf_matrix(y_test, y_pred, title=f"Confusion Matrix - {best_model_name}")
    print(f"🔝 Meilleur modèle : {best_model_name} avec un {sort_metric} de {best_model_score:.4f}")

    return results_df

df_results = compare_models(X_train_scaled, X_test_scaled, y_train, y_test, models)


In [ ]:
f1_before = df_results.iloc[0]["F1-class1"]
macro_before = df_results.iloc[0]["F1-macro"]
weighted_before = df_results.iloc[0]["F1-weighted"]
print(f1_before, macro_before, weighted_before)

### Hyper-parameter tuning

In [ ]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 20]
}

grid = GridSearchCV(RandomForestClassifier(), param_grid, cv=3, scoring='f1_weighted')
grid.fit(X_train_scaled, y_train)

best_model = grid.best_estimator_
print("Meilleurs paramètres :", grid.best_params_)

y_pred_best = best_model.predict(X_test_scaled)
print(classification_report(y_test, y_pred_best))

### Clarification and interpretation of results

# Definition of the metric used

The F1 score is a harmonic mean between precision and recall:

F1 score = 2* (Precision * Recall)/(Precision + Recall)
 
Precision: among the predicted positive cases, how many are actually positive?

Recall (or sensitivity): among the actual positive cases, how many are correctly detected?

Accuracy (86%): The model correctly predicts 86% of total cases. This remains a very good predictive model.

Class 0 (income ≤50K):

Very well recognised with 88% precision and 95% recall.

The 95% recall means that very few people belonging to class 0 are assigned to class 1.

This means that the model is very good at identifying low-income individuals.

Class 1 (income >50K):

Less well recognised: 77% precision but 59% recall.

The 59% recall means that many people earning >£50K are not detected as such by the model.

The F1-score for class 1 (0.67) remains acceptable, but could be improved.

# Interpretation of entities Macro vs Weighted average

Macro vs Weighted Average: The difference between macro avg and weighted avg indicates an imbalance between classes, which remains acceptable, despite the fact that it could be greatly improved (by rebalancing the classes).

Macro average: unweighted average of the scores for each class.

Does not take into account the frequency of the classes.

Reflects the average performance per class.

Weighted average: average weighted by the number of examples in each class.

Gives more weight to the majority class.

Better reflects the overall accuracy of the model.

### Rebalancing classes

In [ ]:
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train_bal, y_train_bal)
y_pred = model.predict(X_test_scaled)

report_after = classification_report(y_test, y_pred, output_dict= True)


plot_conf_matrix(y_test, y_pred, title=f"Confusion Matrix - {model}")

In [ ]:
# Metrics
f1_after = report_after["1"]["f1-score"]

macro_after = report_after['macro avg']['f1-score']

weighted_after = report_after['weighted avg']['f1-score']

In [ ]:
delta_f1 = (f1_after - f1_before) / f1_before

imbalance_before = abs(macro_before - weighted_before)
imbalance_after = abs(macro_after - weighted_after)

improvement = (imbalance_before - imbalance_after) / imbalance_before
gain_macro = (macro_after - macro_before) / macro_before


print(f"F1-score classe 1 - avant SMOTE : {f1_before:.4f}, après SMOTE : {f1_after:.4f}")
print(f"F1-score macro avg - avant : {macro_before:.4f}, après : {macro_after:.4f}")
print(f"F1-score weighted avg - avant : {weighted_before:.4f}, après : {weighted_after:.4f}")
print(f"Réduction du déséquilibre (macro vs weighted) : {improvement:.2%}")
print(f"Amélioration du gain global : {gain_macro:.2%}")



# Interpretation: 
Several results are interesting:

- The F1 score for class 1 remains virtually unchanged (0.668 to 0.679), which implies that there is no real improvement in the detection of the minority class.

- The gap between Macro-avg and Weighted-avg is narrowing, which is consistent with the rebalancing of classes. Indeed, with the reduction in imbalance amounting to 9.49%, the model treats the two classes more fairly.

- Improvement in overall gain: class rebalancing did not affect the overall gain. In fact, rebalancing negatively affected accuracy on the majority class as much as it increased accuracy on the minority class, which does not improve the overall gain.

### Visualisation of the impact of class size increases

will be upgraded soon

In [ ]:
# # Définition de la nouvelle fonction de séparation des classes
# def income_group(income):
#     if income < 30000:
#         return "A (<30k)"
#     elif income < 50000:
#         return "B (30k-50k)"
#     elif income < 100000:
#         return "C (50k-100k)"
#     else:
#         return "D (>100k)"
    
# # Importation de la matrice de départ    
# columns = ["age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
#            "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
#            "hours-per-week", "native-country", "income"]

# df = pd.read_csv("../data/raw/adult.data", names=columns, na_values=" ?", skipinitialspace=True)
# df.dropna(inplace=True)
# df.reset_index(drop=True, inplace=True)
# df["income_group"] = df["income"].apply(income_group)


In [ ]:
# # Création de la figure d'affichage des corrélations
# plt.figure(figsize=(10, 12))

# # Création des corrélations
# correlations = df_encoded.corr(numeric_only=True)[["income"]].sort_values(by="income", ascending=False)

# # Plot de la heatmap des corrélations
# sns.heatmap(correlations, annot=True, cmap="coolwarm", fmt=".2f")
# plt.title("Corrélation des variables avec le revenu (>50K)")
# plt.show()

In [ ]:
# # Définition des variables encodés
# X = df_encoded.drop("income", axis=1)
# y = df_encoded["income"]

# # Partition des varaibles de test et d'entrainement
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# # Normalisation des données
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

In [ ]:
# df_results = compare_models(X_train_scaled, X_test_scaled, y_train, y_test, models)